# KazGPT V5 — Training on Google Colab A100

**Модель:** Qwen2.5-7B-Instruct + QLoRA  
**Датасет:** KazQAD + kk-Wikipedia + **Aya KZ** + **CodeAlpaca KZ** + synthetic (~40k примеров)  
**Улучшения V5:** диалог/greeting, Python код, логика/рассуждения, эмодзи 😊  
**Время:** ~2.5–3 часа на A100 40GB  

### Перед запуском:
1. Runtime → Change runtime type → **A100 GPU**
2. Вставь свой HuggingFace токен в ячейку #3 (`HF_TOKEN`)
3. Запусти все ячейки по порядку (Runtime → Run all)

In [ ]:
# ─── Ячейка 1: Установка зависимостей + монтирование Drive ───────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/KazGPT_V5'
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/adapter',     exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/gguf',        exist_ok=True)
print(f'Drive path: {DRIVE_PATH}')

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q transformers datasets accelerate peft trl bitsandbytes huggingface_hub sentencepiece
print('Packages installed.')

In [ ]:
# ─── Ячейка 2: Проверка GPU ───────────────────────────────────────────────────
import torch

gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok = torch.cuda.is_bf16_supported()

print(f'GPU   : {gpu}')
print(f'VRAM  : {vram_gb:.1f} GB')
print(f'BF16  : {bf16_ok}')

if vram_gb < 20:
    print('\n⚠️  VRAM < 20GB — смени runtime на A100.')
else:
    print('\n✅  Готово к обучению 7B модели.')

In [ ]:
# ─── Ячейка 3: Конфигурация ────────────────────────────────────────────────────
# ⚠️  ЗАПОЛНИ HF_TOKEN: https://huggingface.co/settings/tokens (read access достаточно)
HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXX'

MODEL_ID    = 'Qwen/Qwen2.5-7B-Instruct'
MAX_SEQ     = 2048

LORA_RANK   = 64
LORA_ALPHA  = 128
LORA_DROP   = 0.05

BATCH_SIZE  = 2
GRAD_ACCUM  = 8
EPOCHS      = 3
LR          = 2e-4
WARMUP      = 0.05

# V4 источники
MAX_KAZQAD      = 12000
MAX_WIKI        = 15000
MAX_MULTIDOMAIN = 5000
# V5 новые источники
MAX_AYA         = 3000   # Aya KZ — диалог, diverse tasks
MAX_CODE        = 2000   # CodeAlpaca — Python код с KZ инструкцией

OUT_DIR         = f'{DRIVE_PATH}/checkpoints'
ADAPTER_DIR     = f'{DRIVE_PATH}/adapter'
GGUF_DIR        = f'{DRIVE_PATH}/gguf'

# V5: эмодзи разрешены в приветствиях/позитивных ответах
SYSTEM_PROMPT = (
    'Сен — KazGPT, Қазақстан үшін жасалған жергілікті жасанды интеллект. '
    'АБСОЛЮТТІ ЕРЕЖЕ: Барлық жауаптарыңды ТЕК ҚАЗАҚ ТІЛІНДЕ жаз. '
    'Сыпайы, кәсіби бол. Эмодзиді тек қуаныш/сәлем сөздерінде қолдан. '
    'Жауап қысқа: 1–4 сөйлем.'
)

print('V5 Config OK.')
print(f'  Model       : {MODEL_ID}')
print(f'  Epochs      : {EPOCHS}')
print(f'  LoRA r      : {LORA_RANK}')
print(f'  V4 data     : KazQAD({MAX_KAZQAD}) + Wiki({MAX_WIKI})')
print(f'  V5 new data : Aya({MAX_AYA}) + Code({MAX_CODE}) + synthetic')

In [ ]:
# ─── Ячейка 4: HuggingFace логин ─────────────────────────────────────────────
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK.')

In [ ]:
# ─── Ячейка 5: Загрузка датасета V5 ──────────────────────────────────────────
# V4: KazQAD + kk-Wikipedia  (фактические знания)
# V5: + Aya KZ               (диалог, разнообразные задачи)
#     + CodeAlpaca KZ         (Python код с казахскими инструкциями)
#     + synthetic             (greeting, logic, reasoning)
import random, json
from datasets import load_dataset
random.seed(42)

def to_chatml(question: str, answer: str) -> dict:
    return {
        'text': (
            f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
            f'<|im_start|>user\n{question.strip()}<|im_end|>\n'
            f'<|im_start|>assistant\n{answer.strip()}<|im_end|>'
        )
    }

def is_valid(q: str, a: str) -> bool:
    return bool(q and a and len(q.strip()) > 5 and len(a.strip()) > 10)

all_records = []

# ══ [1/6] KazQAD ════════════════════════════════════════════════
print('=> [1/6] KazQAD ...')
try:
    kqd = load_dataset('issai/kazqad', split='train')
    kqd_records = []
    for item in kqd:
        q = item.get('question') or item.get('query') or ''
        ans = item.get('answers') or item.get('answer') or ''
        if isinstance(ans, dict):
            texts = ans.get('text') or []
            a = texts[0] if texts else ''
        elif isinstance(ans, list):
            a = ans[0] if ans else ''
        else:
            a = str(ans)
        if is_valid(q, a):
            kqd_records.append(to_chatml(q, a))
        if len(kqd_records) >= MAX_KAZQAD:
            break
    all_records += kqd_records
    print(f'   [+] {len(kqd_records)} KazQAD')
except Exception as e:
    print(f'   [!] KazQAD: {e}')

# ══ [2/6] kk-Wikipedia ══════════════════════════════════════════
print('=> [2/6] kk-Wikipedia ...')
try:
    wiki = load_dataset('wikimedia/wikipedia', '20231101.kk',
                        split='train', streaming=True, trust_remote_code=True)
    wiki_records = []
    templates = [
        '{title} деген не?',
        '{title} туралы қысқаша айтып бер.',
        '{title} жайында мәлімет бер.',
        '{title} дегеніміз не?',
    ]
    for item in wiki:
        title = (item.get('title') or '').strip()
        text  = (item.get('text')  or '').strip()
        if not title or not text:
            continue
        first = text.split('.')[0].strip()
        if len(first) < 30 or len(first) > 500:
            continue
        q = random.choice(templates).format(title=title)
        wiki_records.append(to_chatml(q, first + '.'))
        if len(wiki_records) >= MAX_WIKI:
            break
    all_records += wiki_records
    print(f'   [+] {len(wiki_records)} Wikipedia')
except Exception as e:
    print(f'   [!] Wikipedia: {e} — пробую multidomain...')
    try:
        md = load_dataset('kz-transformers/multidomain-kazakh-dataset',
                          split='train', streaming=True)
        md_records = []
        for item in md:
            text = (item.get('text') or item.get('content') or '').strip()
            if len(text) < 100:
                continue
            mid = len(text) // 2
            md_records.append(to_chatml(text[:mid], text[mid:]))
            if len(md_records) >= MAX_MULTIDOMAIN:
                break
        all_records += md_records
        print(f'   [+] {len(md_records)} multidomain')
    except Exception as e2:
        print(f'   [!] multidomain: {e2}')

# ══ [3/6] Aya Dataset (KZ) — NEW V5 ════════════════════════════
print('=> [3/6] Aya Dataset (Kazakh) — NEW V5 ...')
try:
    aya = load_dataset('CohereForAI/aya_dataset', split='train')
    aya_kaz = aya.filter(lambda x: x.get('language') in ('kaz', 'Kazakh', 'kk'))
    aya_records = []
    for item in aya_kaz:
        q = (item.get('inputs') or '').strip()
        a = (item.get('targets') or '').strip()
        if is_valid(q, a):
            aya_records.append(to_chatml(q, a))
        if len(aya_records) >= MAX_AYA:
            break
    all_records += aya_records
    print(f'   [+] {len(aya_records)} Aya KZ')
except Exception as e:
    print(f'   [!] Aya: {e}')

# ══ [4/6] CodeAlpaca → KZ инструкции — NEW V5 ══════════════════
print('=> [4/6] CodeAlpaca → KZ instructions — NEW V5 ...')
try:
    code_ds = load_dataset('sahil2801/CodeAlpaca-20k', split='train')
    kaz_code_templates = [
        '{instr} — Python кодын жаз.',
        'Берілген тапсырма: {instr}. Кодты жаз.',
        '{instr} үшін функция жаз.',
        'Бағдарлама жаз: {instr}.',
        '{instr} — программалау есебін шеш.',
    ]
    code_records = []
    for item in code_ds:
        instr  = (item.get('instruction') or '').strip()
        output = (item.get('output') or '').strip()
        if not instr or not output or len(output) < 20:
            continue
        kaz_instr = random.choice(kaz_code_templates).format(instr=instr)
        code_records.append(to_chatml(kaz_instr, output))
        if len(code_records) >= MAX_CODE:
            break
    all_records += code_records
    print(f'   [+] {len(code_records)} code pairs')
except Exception as e:
    print(f'   [!] CodeAlpaca: {e}')

# ══ [5/6] Synthetic Greetings — NEW V5 ═════════════════════════
print('=> [5/6] Synthetic greetings ...')
greetings = [
    ('Сәлем!',              'Сәлем! 😊 Сізге қалай көмектесе аламын?'),
    ('Қалайсыз?',           'Жақсымын, рахмет! Сіз қалайсыз?'),
    ('Қалың қалай?',        'Жақсымын! 😊 Сізге қандай сұрақ бар?'),
    ('Сәлеметсіз бе!',      'Сәлеметсіз! Сізге қалай көмектесейін?'),
    ('Кешке жарық!',        'Кешке жарық! Сізге қандай сұрақ бар?'),
    ('Қайырлы таң!',        'Қайырлы таң! 🌅 Бүгін қалай көмектесе аламын?'),
    ('Привет!',             'Сәлем! Қазақ тілінде сөйлесейік — қалайсыз? 😊'),
    ('Привет, как дела?',   'Сәлем! Жақсымын. Сізге қалай көмектесе аламын?'),
    ('Hello!',              'Сәлем! KazGPT-ке қош келдіңіз 🇰🇿 Қазақша сөйлесейік!'),
    ('Рахмет!',             'Өтінемін! 😊 Басқа сұрағыңыз болса айтыңыз.'),
    ('Сау бол!',            'Сау болыңыз! 👋 Тағы кіріңіз.'),
    ('Жақсы ма?',           'Жақсы, рахмет! Сізге қалай көмектесейін?'),
    ('Бар болыңыз!',        'Рахмет! Сау болыңыз! 👋'),
    ('Өте жақсы!',          'Керемет! 🎉 Сізге тағы да қалай көмектесе аламын?'),
    ('Не жаңалық?',         'Бәрі жақсы! 😊 Сізде не жаңалық?'),
    ('Қош келдіңіз!',       'Рахмет! 🙏 Сізге қалай көмектесе аламын?'),
]

# ══ [6/6] Synthetic Reasoning/Logic — NEW V5 ═══════════════════
print('=> [6/6] Synthetic reasoning + logic ...')
reasoning = [
    ('Егер барлық адамдар өлімді болса, Сократ адам болса, Сократ өлімді ме?',
     'Иә, Сократ өлімді. Себебі: 1) Барлық адамдар өлімді. 2) Сократ — адам. 3) Демек, Сократ та өлімді. Бұл классикалық силлогизм деп аталады.'),
    ('Судың қайнау температурасы қанша?',
     'Су теңіз деңгейінде 100°C-та қайнайды.'),
    ('Айда ауа бар ма?',
     'Жоқ, Айда атмосфера жоқ, сондықтан ауа да жоқ.'),
    ('1 километрде неше метр бар?',
     '1 километрде 1000 метр бар.'),
    ('Ең үлкен мұхит қайсы?',
     'Тынық мұхит — жер шарындағы ең үлкен мұхит.'),
    ('100 ішінде жай сандарды атап бер.',
     '2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97.'),
    ('Неліктен аспан көк?',
     'Аспан Рэлей шашырауы салдарынан көк болып көрінеді: қысқа толқынды көк сәуле ұзын толқынды қызыл сәулеге қарағанда атмосферада көп шашырайды.'),
    ('Қазақстан қай жылы тәуелсіздік алды?',
     'Қазақстан 1991 жылы 16 желтоқсанда тәуелсіздік жариялады.'),
    ('Жылдамырақ: жаяу жүру немесе велосипед?',
     'Велосипед жылдамырақ — дөңгелек қозғалысы аяқтың қадамынан әлдеқайда тиімді.'),
    ('Python-да Фибоначчи санын есептейтін функция жаз.',
     'def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)'),
    ('Python-да 1-ден 10-ға дейін санды экранға шығар.',
     'for i in range(1, 11):\n    print(i)'),
    ('Python-да тізімнің ең үлкен элементін тап.',
     'def find_max(lst):\n    return max(lst)\n\n# Мысал:\nprint(find_max([3, 1, 4, 1, 5, 9, 2, 6]))  # 9'),
]

synthetic_all = [to_chatml(q, a) for q, a in greetings + reasoning]
all_records += synthetic_all
print(f'   [+] {len(greetings)} greetings + {len(reasoning)} reasoning')

# ══ ФИНАЛЬНЫЙ ДАТАСЕТ ════════════════════════════════════════════
random.shuffle(all_records)
split = int(len(all_records) * 0.95)
train_data = all_records[:split]
valid_data = all_records[split:]

print(f'\n{"═"*45}')
print(f'V5 Датасет готов:')
print(f'  Всего   : {len(all_records)}')
print(f'  Train   : {len(train_data)}')
print(f'  Valid   : {len(valid_data)}')
print(f'{"═"*45}')

In [ ]:
# ─── Ячейка 6: Загрузка модели через unsloth ─────────────────────────────────
from unsloth import FastLanguageModel
import torch

print(f'Загружаю {MODEL_ID} с 4-bit QLoRA ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ,
    dtype           = None,
    load_in_4bit    = True,
    token           = HF_TOKEN,
)

vram_used = torch.cuda.memory_allocated() / 1e9
print(f'Модель загружена. VRAM занято: {vram_used:.1f} GB')

In [ ]:
# ─── Ячейка 7: LoRA адаптер ───────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r               = LORA_RANK,
    target_modules  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = LORA_DROP,
    bias            = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state    = 42,
    use_rslora      = False,
)
model.print_trainable_parameters()

In [ ]:
# ─── Ячейка 8: Обучение ───────────────────────────────────────────────────────
import time
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

train_ds = Dataset.from_list(train_data)
valid_ds = Dataset.from_list(valid_data)

bf16 = torch.cuda.is_bf16_supported()

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_ds,
    eval_dataset    = valid_ds,
    dataset_text_field = 'text',
    max_seq_length  = MAX_SEQ,
    packing         = True,
    args = TrainingArguments(
        output_dir                  = OUT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio                = WARMUP,
        learning_rate               = LR,
        lr_scheduler_type           = 'cosine',
        fp16                        = not bf16,
        bf16                        = bf16,
        evaluation_strategy         = 'epoch',
        save_strategy               = 'epoch',
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        logging_steps               = 25,
        report_to                   = 'none',
        seed                        = 42,
        dataloader_num_workers      = 2,
    ),
)

print('═' * 50)
print('V5 Обучение начинается...')
print(f'Train: {len(train_ds)} | Valid: {len(valid_ds)}')
print('═' * 50)

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 3600

print(f'\n✅  V5 обучение завершено за {elapsed:.2f} ч')
print(f'   Best eval_loss: {trainer.state.best_metric:.4f}')

In [ ]:
# ─── Ячейка 9: Сохранение LoRA адаптера ──────────────────────────────────────
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'V5 адаптер сохранён: {ADAPTER_DIR}')
!ls -lh {ADAPTER_DIR}

In [ ]:
# ─── Ячейка 10: Проверка качества V5 ─────────────────────────────────────────
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def ask(question: str) -> str:
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{question}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = 512,
            temperature        = 0.27,
            top_p              = 0.85,
            repetition_penalty = 1.15,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()

test_questions = [
    # Greeting (V4 провалил)
    'Сәлем! Қалың қалай?',
    'Привет, как дела?',
    # Факты
    'Қазақстанның астанасы қандай?',
    'Нейрондық желі деген не?',
    # Логика (V4 провалил)
    'Егер барлық адамдар өлімді болса, Сократ адам болса, Сократ өлімді ме?',
    # Код (V4 провалил)
    'Python-да Фибоначчи санын есептейтін функция жаз.',
    # Смешанный
    'Machine learning деген не, қазақша түсіндір.',
]

print('═' * 50)
print('Проверка качества V5:')
print('═' * 50)
for q in test_questions:
    a = ask(q)
    print(f'Q: {q}')
    print(f'A: {a}')
    print('─' * 40)

In [ ]:
# ─── Ячейка 11: Экспорт в GGUF для Ollama ────────────────────────────────────
print('Экспортирую V5 в GGUF Q4_K_M...')
print('(Это займёт 5–15 минут)')

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method = 'q4_k_m',
)

gguf_files = !ls -lh {GGUF_DIR}/*.gguf 2>/dev/null
print('GGUF файлы:')
for f in gguf_files:
    print(f'  {f}')
print(f'\n✅  V5 готово! Скачай файл из: {GGUF_DIR}')

## После обучения — подключение KazGPT V5

### 1. Скачай GGUF из Google Drive
Папка: `KazGPT_V5/gguf/` → файл `*.gguf` (~4–5 GB)

### 2. Создай Modelfile для Ollama
```
FROM C:\путь\к\файлу\kazgpt-v5.gguf

PARAMETER temperature 0.27
PARAMETER top_p 0.85
PARAMETER repeat_penalty 1.15
PARAMETER num_predict 512
```

### 3. Зарегистрируй модель в Ollama
```bash
ollama create kazgpt-v5 -f Modelfile
ollama run kazgpt-v5 "Сәлем!"
```

### 4. Обнови application.yml
```yaml
kazgpt:
  models:
    base:
      url: http://localhost:11434
      name: kazgpt-v5
      runtime: ollama
```

### 5. Перезапусти Spring Boot
```bash
cd backend && mvn spring-boot:run
```